In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

ModuleNotFoundError: No module named 'imblearn'

In [ ]:
#configuracion de carpetas 
CARPETA_SALIDA = "Graficos_Informe"

#Creamos la carpeta automáticamente
if not os.path.exists(CARPETA_SALIDA):
    os.makedirs(CARPETA_SALIDA)
    print(f"Carpeta creada: {CARPETA_SALIDA}")
else:
    print(f"La carpeta '{CARPETA_SALIDA}' ya existe. Los gráficos se guardarán ahí.")

#paleta
COLORES = {
    'Verde_Fuerte': '#74b404', 
    'Verde_Claro':  '#cdfc7d', 
    'Rojo_Corp':    '#aa044c', 
    'Morado_Os':    '#876784',
    'Morado_Cl':    '#b09eae'
}

#Creamos funcion para guardar aqui los graficos
def guardar_grafico(fig, nombre_archivo):
    """
    Guarda el gráfico en HTML (interactivo) y PNG (estático) 
    dentro de la carpeta organizada automáticamente.
    """
    # Guardar HTML (Interactivo)
    ruta_html = os.path.join(CARPETA_SALIDA, f"{nombre_archivo}.html")
    fig.write_html(ruta_html)
    
    print(f"Gráfico guardado: {ruta_html}")

#Carga de datos
def cargar_y_preparar_datos(ruta_archivo):
    print(f"Cargando datos desde: {ruta_archivo}")
    try:
        df = pd.read_excel(ruta_archivo)
        
        df_viv = df[df['Proposito'].astype(str).str.contains('Vivienda', case=False, na=False)].copy()
        df_viv['Impago_Label'] = df_viv['Impago'].map({0: 'Pagado', 1: 'Impago'})
        
        if 'Posesion_Hipoteca' in df_viv.columns:
            df_viv['Tiene_Hipoteca'] = df_viv['Posesion_Hipoteca'].map({0: 'No tiene', 1: 'Sí tiene'})
            
        col_fiador = 'Fiador' if 'Fiador' in df_viv.columns else 'Cofirmante'
        df_viv['Fiador_Label'] = df_viv[col_fiador].map({0: 'Sin Fiador', 1: 'Con Fiador'})
            
        return df_viv
    except Exception as e:
        print(f"Error: {e}")
        return None

#Cargamos
ruta_real = os.path.join('..', 'Datos', 'Originales', 'información_préstamos.xlsx')
df_viv = cargar_y_preparar_datos(ruta_real)

La carpeta 'Graficos_Informe' ya existe. Los gráficos se guardarán ahí.
Cargando datos desde: ..\Datos\Originales\información_préstamos.xlsx


In [ ]:
variables_cluster = [
    "Ingresos",
    "Monto_Inicial",
    "Duracion",
    "Edad"
]

X_cluster = df_viv[variables_cluster]

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=20
)

df_viv["cluster"] = kmeans.fit_predict(X_cluster_scaled)

In [ ]:
# ======================================================
# 3. VARIABLES PARA EL MODELO PREDICTIVO
# ======================================================
vars_predictoras = [
    'Ingresos', 'Monto_Inicial', 'Edad', 'Scoring_Crediticio', 
    'Meses_Empleo', 'Num_Creditos', 'Ratio_Deuda_Ingresos', 
    'Personas_Cargo', 'Ratio_Interes', 'Duracion',
    'cluster'   #cluster como variable explicativa
]

vars_texto = [
    'Estado_Civil',
    'Estudios',
    'Tipo_Jornada_Laboral'
]


# ======================================================
# 4. DATASET FINAL DEL MODELO
# ======================================================
df_model = df_viv[vars_predictoras + vars_texto + ['Impago']].copy()
df_model = df_model.dropna()

# One-Hot Encoding (texto + cluster)
df_model = pd.get_dummies(
    df_model,
    columns=vars_texto + ['cluster'],
    drop_first=True
)


# ======================================================
# 5. TRAIN / TEST SPLIT
# ======================================================
X = df_model.drop('Impago', axis=1)
y = df_model['Impago']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42
)

print(f"🧠 Datos preparados. Entrenando con {len(X_train)} clientes...")



acc_rf = accuracy_score(y_test, y_pred_rf)

**MODELO 1: Desbalanceado + SIN PCA**

In [ ]:
scaler_1 = StandardScaler()
X_train_scaled = scaler_1.fit_transform(X_train)
X_test_scaled  = scaler_1.transform(X_test)

modelo_1 = LogisticRegression(max_iter=1000, random_state=42)
modelo_1.fit(X_train_scaled, y_train)

y_pred_1 = modelo_1.predict(X_test_scaled)

evaluar_modelo("Modelo 1: Desbalanceado + Sin PCA", y_test, y_pred_1)

**MODELO 2: Desbalanceado + CON PCA**

In [ ]:
scaler_2 = StandardScaler()
X_train_scaled = scaler_2.fit_transform(X_train)
X_test_scaled  = scaler_2.transform(X_test)

pca_2 = PCA(n_components=0.95, random_state=42)
X_train_pca = pca_2.fit_transform(X_train_scaled)
X_test_pca  = pca_2.transform(X_test_scaled)

modelo_2 = LogisticRegression(max_iter=1000, random_state=42)
modelo_2.fit(X_train_pca, y_train)

y_pred_2 = modelo_2.predict(X_test_pca)

evaluar_modelo("Modelo 2: Desbalanceado + Con PCA", y_test, y_pred_2)


**MODELO 3: Balanceado (SMOTE) + SIN PCA**

In [ ]:
scaler_3 = StandardScaler()
X_train_scaled = scaler_3.fit_transform(X_train)
X_test_scaled  = scaler_3.transform(X_test)

smote_3 = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote_3.fit_resample(X_train_scaled, y_train)

modelo_3 = LogisticRegression(max_iter=1000, random_state=42)
modelo_3.fit(X_train_sm, y_train_sm)

y_pred_3 = modelo_3.predict(X_test_scaled)

evaluar_modelo("Modelo 3: Balanceado (SMOTE) + Sin PCA", y_test, y_pred_3)


**MODELO 4: Balanceado (SMOTE) + CON PCA**

In [ ]:
scaler_4 = StandardScaler()
X_train_scaled = scaler_4.fit_transform(X_train)
X_test_scaled  = scaler_4.transform(X_test)

smote_4 = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote_4.fit_resample(X_train_scaled, y_train)

pca_4 = PCA(n_components=0.95, random_state=42)
X_train_pca = pca_4.fit_transform(X_train_sm)
X_test_pca  = pca_4.transform(X_test_scaled)

modelo_4 = LogisticRegression(max_iter=1000, random_state=42)
modelo_4.fit(X_train_pca, y_train_sm)

y_pred_4 = modelo_4.predict(X_test_pca)

evaluar_modelo("Modelo 4: Balanceado (SMOTE) + Con PCA", y_test, y_pred_4)